# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✗ OpenAI API Key가 없습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [3]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "table1": "../datasets/professors_rows.csv",
    "table2": "../datasets/courses_rows.csv"


    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 table1 테이블

행 수: 5
컬럼: ['name', 'degree', 'department', 'major', 'phone', 'location']

첫 5개 행:
  name degree department                                             major  \
0  이유진     박사      전자공학과                                            전자파적합성   
1  이준하     박사      전자공학과                                        반도체재료 및 공정   
2  이흥주     박사      전자공학과                                              전자공학   
3  정민철     박사      전자공학과                                      컴퓨터비전 및 인공지능   
4  조준희     박사      전자공학과  Optoelectronics / Energy Nanomaterials & Devices   

          phone     location  
0  041-550-5413  한누리관 (I606)  
1  041-550-5362  한누리관 (I607)  
2  041-550-5360  한누리관 (I608)  
3  041-550-5361  한누리관 (I611)  
4  041-550-5134  한누리관 (I610)  

데이터 타입:
name          str
degree        str
department    str
major         str
phone         str
location      str
dtype: object


📋 table2 테이블

행 수: 37
컬럼: ['grade', 'semester', 'course_type', 'course_code', 'course_name', 'credit', 'hours', 'pr

## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [4]:
# TODO: 각 테이블의 주요 통계를 확인하세요
# 예시:
# - 특정 컬럼의 고유값 개수
# - 카테고리별 데이터 분포
# - 결측치 확인

for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    print(df.info())

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # TODO: 팀 데이터에 맞는 추가 탐색 코드를 작성하세요
    # 카테고리별 데이터 분포 확인
    print("\n[카테고리 분포]")
    if table_name == 'courses':
        print("\n1. 학년별 과목 수:")
        print(df['grade'].value_counts())
        print("\n2. 이수구분(전심/전선) 수:")
        print(df['course_type'].value_counts())
        print("\n3. 학기별 과목 수:")
        print(df['semester'].value_counts())
        
    elif table_name == 'professors':
        print("\n1. 세부전공 목록 및 인원:")
        print(df['major'].value_counts())
        print("\n2. 연구실 위치 분포:")
        print(df['location'].value_counts())


📊 table1 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        5 non-null      str  
 1   degree      5 non-null      str  
 2   department  5 non-null      str  
 3   major       5 non-null      str  
 4   phone       5 non-null      str  
 5   location    5 non-null      str  
dtypes: str(6)
memory usage: 813.0 bytes
None

[결측치]
결측치 없음

[카테고리 분포]

📊 table2 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   grade           37 non-null     str  
 1   semester        37 non-null     str  
 2   course_type     37 non-null     str  
 3   course_code     37 non-null     str  
 4   course_name     37 non-null     str  
 5   credit          37 non-null     int64
 6   hours           37 non-null     int64
 7   p

## 3. Supabase PostgreSQL 연결

In [5]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

C:\Users\leemb\AppData\Local\Temp\ipykernel_13476\1442992373.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['courses', 'professors']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [6]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE courses (
	grade TEXT, 
	semester TEXT, 
	course_type TEXT, 
	course_code TEXT, 
	course_name TEXT, 
	credit BIGINT, 
	hours BIGINT, 
	professor_name TEXT
)

/*
3 rows from courses table:
grade	semester	course_type	course_code	course_name	credit	hours	professor_name
1학년	1학기	1전선	HBJN2149	회로망론Ⅰ	3	3	이흥주
1학년	1학기	1전선	HBJN2168	회로망론II	3	3	이흥주
1학년	1학기	1전선	HBJW0002	컴퓨터프로그래밍Ⅰ(SW)	3	3	정민철
*/


CREATE TABLE professors (
	name TEXT, 
	degree TEXT, 
	department TEXT, 
	major TEXT, 
	phone TEXT, 
	location TEXT
)

/*
3 rows from professors table:
name	degree	department	major	phone	location
이흥주	박사	전자공학과	전자공학	041-550-5360	한누리관 (I608)
정민철	박사	전자공학과	컴퓨터비전 및 인공지능	041-550-5361	한누리관 (I611)
이준하	박사	전자공학과	반도체재료 및 공정	041-550-5362	한누리관 (I607)
*/


courses 테이블 샘플:
[('1학년', '1학기', '1전선', 'HBJN2149', '회로망론Ⅰ', 3, 3, '이흥주'), ('1학년', '1학기', '1전선', 'HBJN2168', '회로망론II', 3, 3, '이흥주'), ('1학년', '1학기', '1전선', 'HBJW0002', '컴퓨터프로그래밍Ⅰ(SW)', 3, 3, '정민철')]

professors 테이블 샘플:
[('이흥주', '박사', '전자공학

## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [7]:
# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = """
SELECT c.grade, c.semester, c.course_type, c.course_code, c.course_name, c.credit, c.hours
FROM courses c
WHERE c.professor_name = '이흥주'
ORDER BY c.grade, c.semester, c.course_code;

"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT c.grade, c.semester, c.course_type, c.course_code, c.course_name, c.credit, c.hours
FROM courses c
WHERE c.professor_name = '이흥주'
ORDER BY c.grade, c.semester, c.course_code;



결과:
[('1학년', '1학기', '1전선', 'HBJN2149', '회로망론Ⅰ', 3, 3), ('1학년', '1학기', '1전선', 'HBJN2168', '회로망론II', 3, 3), ('1학년', '2학기', '1전선', 'HBJN2149', '회로망론Ⅰ', 3, 3), ('1학년', '2학기', '1전선', 'HBJN2168', '회로망론II', 3, 3), ('2학년', '1학기', '1전심', 'HBJW0046', '기초회로망실험', 3, 3), ('3학년', '1학기', '1전선', 'HBJN2160', '마이크로프로세서(전자공학과)', 3, 3), ('3학년', '2학기', '1전심', 'HBJW0047', '응용전자회로실험', 3, 3), ('4학년', '1학기', '1전심', 'HBJW0011', '캡스톤디자인I(전자공학과)', 3, 3), ('4학년', '2학기', '1전선', 'HBJW0050', '캡스톤디자인II(전자공학과)', 3, 3)]


In [8]:
query

"\nSELECT c.grade, c.semester, c.course_type, c.course_code, c.course_name, c.credit, c.hours\nFROM courses c\nWHERE c.professor_name = '이흥주'\nORDER BY c.grade, c.semester, c.course_code;\n\n"

In [9]:
# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = """
SELECT *
FROM courses;

"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT *
FROM courses;



결과:
[('1학년', '1학기', '1전선', 'HBJN2149', '회로망론Ⅰ', 3, 3, '이흥주'), ('1학년', '1학기', '1전선', 'HBJN2168', '회로망론II', 3, 3, '이흥주'), ('1학년', '1학기', '1전선', 'HBJW0002', '컴퓨터프로그래밍Ⅰ(SW)', 3, 3, '정민철'), ('1학년', '1학기', '1전선', 'HBJW0003', '공학수학Ⅰ(PBL)', 3, 3, '조준희'), ('1학년', '1학기', '1전선', 'HBJW0004', '컴퓨터프로그래밍II', 3, 3, '정민철'), ('1학년', '1학기', '1전선', 'HBJW0034', '공학수학Ⅱ(PBL)', 3, 3, '조준희'), ('1학년', '1학기', '1전선', 'HBJW0056', '전공체험(전자공학과)', 2, 2, '이유진'), ('1학년', '2학기', '1전선', 'HBJN2149', '회로망론Ⅰ', 3, 3, '이흥주'), ('1학년', '2학기', '1전선', 'HBJN2168', '회로망론II', 3, 3, '이흥주'), ('1학년', '2학기', '1전선', 'HBJW0002', '컴퓨터프로그래밍Ⅰ(SW)', 3, 3, '정민철'), ('1학년', '2학기', '1전선', 'HBJW0003', '공학수학Ⅰ(PBL)', 3, 3, '조준희'), ('1학년', '2학기', '1전선', 'HBJW0004', '컴퓨터프로그래밍II', 3, 3, '정민철'), ('1학년', '2학기', '1전선', 'HBJW0034', '공학수학Ⅱ(PBL)', 3, 3, '조준희'), ('1학년', '2학기', '1전선', 'HBJW0056', '전공체험(전자공학과)', 2, 2, '이유진'), ('2학년', '1학기', '1전심', 'HBJN2022', '디지털공학(PBL)', 3, 3, '이유진'), ('2학년', '1학기', '1전심', 'HBJW0027', '전자기학',

In [2]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT, SUM, AVG 등 사용

aggregation_query = """
-- TODO: 집계 쿼리를 작성하세요
-- 예시:
-- SELECT category, COUNT(*) as count, AVG(price) as avg_price
-- FROM products
-- GROUP BY category
-- ORDER BY count DESC;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

-- TODO: 집계 쿼리를 작성하세요
-- 예시:
-- SELECT category, COUNT(*) as count, AVG(price) as avg_price
-- FROM products
-- GROUP BY category
-- ORDER BY count DESC;


결과:
쿼리 실행 오류: name 'db' is not defined


## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    당신은 SQL 전문가입니다.
    사용자의 질문을 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [ ]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "이흥주 교수 강의 알려줘"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: 이흥주 교수 강의 알려줘



RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [ ]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 전자공학과 조교입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문에 자연스럽게 답변하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    정확하지 않은 내용은 추측해서 답하지 마세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [ ]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "이흥주 교수님의 1학년 수업은?"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 이흥주 교수님의 1학년 수업은?


[1] SQL 생성 중...


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [ ]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "YOUR_QUESTION_1_HERE",
    "YOUR_QUESTION_2_HERE",
    "YOUR_QUESTION_3_HERE",
    "YOUR_QUESTION_4_HERE",
    "YOUR_QUESTION_5_HERE"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용